# Clase 7 – Combinación y Fusión de DataFrames
### Limpieza y preparación de datos

**Temas:** `pd.concat()` · `pd.merge()` · `inner` / `left` / `right` / `outer` · `left_on` / `right_on`

---

In [ ]:
import pandas as pd
print(f'pandas {pd.__version__} cargado ✅')

---
## Slide 3 – Desafío Inicial: Pedidos y Clientes

Dos archivos separados que deben combinarse para análisis de comportamiento por zona.
Problema: no todos los clientes hicieron pedidos, y hay pedidos sin cliente vinculado.

**Archivos:** `clientes.csv` · `pedidos.csv`

In [ ]:
# ── Slide 3: Carga del desafío inicial ───────────────────────────────────────

import pandas as pd
clientes = pd.read_csv('clientes.csv')
pedidos  = pd.read_csv('pedidos.csv')

print(f'clientes.csv: {len(clientes)} filas | {list(clientes.columns)}')
print(f'pedidos.csv:  {len(pedidos)} filas | {list(pedidos.columns)}')
print()
print(clientes.head(5))
print()
print(pedidos.head(5))

---
## Slides 5–7 – Concatenación de DataFrames con `pd.concat()`

Une DataFrames secuencialmente (misma estructura, apilando filas).

In [ ]:
# ── Slide 6: Dos DataFrames con la misma estructura ───────────────────────────

import pandas as pd
enero   = pd.DataFrame({'producto': ['A', 'B'], 'ventas': [100, 150]})
febrero = pd.DataFrame({'producto': ['A', 'B'], 'ventas': [110, 160]})

In [ ]:
# ── Slide 7: Resultado de la concatenación ────────────────────────────────────

ventas_total = pd.concat([enero, febrero], ignore_index=True)
print(ventas_total)
# Resultado:
# producto ventas
# 0    A    100
# 1    B    150
# 2    A    110
# 3    B    160

---
## Slide 8 – Parámetros Clave de `concat()`

| Parámetro | Descripción |
|-----------|-------------|
| `axis=0` | Une por filas (default) |
| `axis=1` | Une por columnas |
| `ignore_index` | Reinicia el índice |
| `keys` | Agrega etiquetas jerárquicas |
| `join` | `outer` conserva todas las columnas; `inner` solo las comunes |

In [ ]:
# ── Slide 8: Parámetros clave — ejemplos ──────────────────────────────────────

# axis=1: unir por columnas
print('axis=1 (por columnas):')
print(pd.concat([enero, febrero], axis=1))

# keys: etiquetas jerárquicas
print('\nkeys (etiquetas jerárquicas):')
print(pd.concat([enero, febrero], keys=['Enero', 'Febrero']))

# join='inner': solo columnas comunes
marzo = pd.DataFrame({'producto': ['C'], 'ventas': [90], 'stock': [5]})
print('\njoin="inner" (solo columnas comunes):')
print(pd.concat([enero, marzo], join='inner', ignore_index=True))

---
## Slide 11 – Merge de DataFrames

`merge()` une DataFrames en base a columnas clave (como JOIN en SQL).

In [ ]:
# ── Slide 11: Sintaxis básica de merge() ──────────────────────────────────────

pd.merge(df1, df2, on='columna_clave', how='tipo_de_union')

---
## Slide 12 – Tipos de Combinación (`how`)

| `how` | Descripción |
|-------|-------------|
| `'inner'` | Solo registros que coincidan en ambas tablas |
| `'left'` | Todos los de la izquierda + coincidencias de la derecha |
| `'right'` | Todos los de la derecha + coincidencias de la izquierda |
| `'outer'` | Todos los registros, coincidan o no |

---
## Slide 13 – Ejemplo Aplicado: Clientes y Pedidos

In [ ]:
# ── Slide 13: Unión de clientes y pedidos (left join) ─────────────────────────
# Pedido con id_cliente=5 no existe en clientes → aparece como NaN

clientes = pd.read_csv('clientes.csv')
pedidos  = pd.read_csv('pedidos.csv')
df = pd.merge(pedidos, clientes, on='id_cliente', how='left')
print(df)
# Resultado esperado:
# id_pedido id_cliente monto nombre
# 0 101 1 5000 Ana
# 1 102 2 7000 Luis
# 2 103 2 3000 Luis
# 3 104 5 2000 NaN

---
## Slides 15–17 – Actividad Guiada: Combinación de Clientes y Pedidos

**Objetivo (Slide 16):** Consolidar clientes y pedidos para análisis comercial.

**Archivos:** `clientes.csv` · `pedidos.csv`

In [ ]:
# ── Slide 17: Cargar los archivos ─────────────────────────────────────────────

clientes = pd.read_csv('clientes.csv')
pedidos  = pd.read_csv('pedidos.csv')

In [ ]:
# ── Slide 17: Concatenar nuevos registros ──────────────────────────────────────

clientes_actualizados = pd.concat([clientes, clientes], ignore_index=True)
print(clientes_actualizados.head())

In [ ]:
# ── Slide 17: Unir con pedidos (left merge) ───────────────────────────────────

df_final = pd.merge(pedidos, clientes, on='id_cliente', how='left')
print(df_final)

In [ ]:
# ── Slide 18: Análisis del resultado ─────────────────────────────────────────

print('=== Análisis del dataset consolidado ===')
print(f'Total de pedidos:                     {len(df_final)}')
print(f'Pedidos con cliente identificado:     {df_final["nombre"].notna().sum()}')
print(f'Pedidos sin cliente (NaN en nombre):  {df_final["nombre"].isna().sum()}')
print()

# Clientes que NO hicieron pedidos
df_outer = pd.merge(pedidos, clientes, on='id_cliente', how='outer')
sin_pedido = df_outer[df_outer['id_pedido'].isna()]
print(f'Clientes sin ningún pedido: {len(sin_pedido)}')
print(sin_pedido[['id_cliente','nombre','region']])

# Monto total por región (solo pedidos con cliente identificado)
print('\nMonto total por región:')
print(df_final.groupby('region')['monto'].sum().sort_values(ascending=False))

In [ ]:
# ── Slide 18: Demostración de los 4 tipos de join ─────────────────────────────

print(f'{"how":>8} | {"Filas resultado":>16} | {"NaN en nombre":>15} | Descripción')
print('-' * 70)
for how in ['inner', 'left', 'right', 'outer']:
    df_h = pd.merge(pedidos, clientes, on='id_cliente', how=how)
    nan_nombre = df_h['nombre'].isna().sum() if 'nombre' in df_h.columns else '-'
    desc = {'inner':'Solo coincidencias', 'left':'Todos los pedidos',
            'right':'Todos los clientes', 'outer':'Todos de ambos'}[how]
    print(f'{how:>8} | {len(df_h):>16} | {str(nan_nombre):>15} | {desc}')

---
## Slides 19–21 – Actividad Autónoma: Productos y Proveedores

**Contexto (Slide 20):** Identificar productos sin proveedor y proveedores sin productos.

**Archivos:** `productos.csv` · `proveedores.csv`

---
**Completa las celdas marcadas con `# 📝 TU CÓDIGO AQUÍ`**

In [ ]:
# ── Slide 21: Cargar los archivos ─────────────────────────────────────────────

productos   = pd.read_csv('productos.csv')
proveedores = pd.read_csv('proveedores.csv')
print(f'productos.csv:   {len(productos)} filas | {list(productos.columns)}')
print(f'proveedores.csv: {len(proveedores)} filas | {list(proveedores.columns)}')
print()
print(productos.head())
print()
print(proveedores)

In [ ]:
# ── Slide 21: Unión izquierda — todos los productos (con o sin proveedor) ──────
# Productos sin proveedor → NaN en columnas de proveedores

# 📝 TU CÓDIGO AQUÍ
# df_left = pd.merge(productos, proveedores, on='id_proveedor', how='left')
# print(df_left)

In [ ]:
# ── Slide 21: ¿Cuántos productos no tienen proveedor asignado? ────────────────

# 📝 TU CÓDIGO AQUÍ
# sin_proveedor = df_left[df_left['nombre_proveedor'].isna()]
# print(f'Productos sin proveedor: {len(sin_proveedor)}')

In [ ]:
# ── Slide 21: Unión exterior — todo de ambos tablas ───────────────────────────
# Proveedores sin productos también aparecen

# 📝 TU CÓDIGO AQUÍ
# df_outer = pd.merge(productos, proveedores, on='id_proveedor', how='outer')
# proveedores_sin_productos = df_outer[df_outer['id_producto'].isna()]
# print(f'Proveedores sin productos: {len(proveedores_sin_productos)}')

In [ ]:
# ── Slide 21: Exportar resultados como CSV ────────────────────────────────────

# 📝 TU CÓDIGO AQUÍ
# df_left.to_csv('productos_con_proveedor.csv', index=False)
# df_outer.to_csv('productos_proveedores_completo.csv', index=False)

In [ ]:
# ── Ejercicio adicional: left_on / right_on (Slide 14) ────────────────────────
# Cuando los nombres de la clave son distintos en cada DataFrame
# Crea un ejemplo donde la clave se llame 'id_prov' en productos y 'id' en proveedores
# y usa left_on='id_prov', right_on='id'

# 📝 TU CÓDIGO AQUÍ

### Preguntas de reflexión y cierre (Slide 18)

1. ¿Qué sucede si un pedido tiene un `id_cliente` que no existe en la tabla de clientes?
2. ¿Qué combinación (`how`) incluiría todos los pedidos, incluso los sin cliente?
3. ¿Cuál sería el efecto de tener claves duplicadas en ambos DataFrames?
4. ¿Cuándo preferirías `concat()` sobre `merge()`?